# Preamble

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer

c:\Users\opens\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Data Processing

In [2]:
lmcache_files = sorted(Path("raw").glob("*.parquet"))
lmcache_df = pd.concat(
    [pd.read_parquet(path) for path in lmcache_files],
    ignore_index=True,
)


lmcache_df["step"] = (
    lmcache_df.groupby("session_id", sort=False).cumcount() + 1
)

print(lmcache_df.columns)


Index(['session_id', 'model', 'input', 'output_length', 'pre_gap', 'step'], dtype='str')


## Input Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B", revision="70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",)
tqdm.pandas()

def normalize_messages(value):
    if isinstance(value, np.ndarray):
        return normalize_messages(value.tolist())
    if isinstance(value, dict):
        return {key: normalize_messages(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [normalize_messages(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value

def count_input_tokens(messages):
    encoded = tokenizer.apply_chat_template(
        normalize_messages(messages),
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=True,
        return_dict=True,
    )
    return len(encoded["input_ids"])

lmcache_df["input_tokens"] = (
    lmcache_df["input"].progress_apply(count_input_tokens)
)

100%|██████████| 24880/24880 [15:49<00:00, 26.21it/s] 


## Below 32k Input + Output Tokens

In [ ]:
total_tokens = (
    lmcache_df["input_tokens"] + lmcache_df["output_length"]
)

session_max_total = total_tokens.groupby(
    lmcache_df["session_id"]
).max()

eligible_sessions = session_max_total[
    session_max_total <= 32768
].index

lmcache_32k = lmcache_df[
    lmcache_df["session_id"].isin(eligible_sessions)
].copy()

In [5]:
output = Path("processed/trace_32k.parquet")
output.parent.mkdir(parents=True, exist_ok=True)

lmcache_32k.to_parquet(output, engine="pyarrow", index=False)